# 黄金价格 vs 战争事件可视化分析

本 Notebook 分析历次战争期间黄金价格的表现。

## 数据源

由于网络限制，推荐以下方式获取数据：

### 方式 1: Investing.com（推荐，数据最全）

1. 访问 [Investing.com 黄金历史数据](https://www.investing.com/commodities/gold-historical-data)
2. 设置日期范围（建议 **1970-01-01 至今**）
3. 点击 **Download Data** 下载 CSV
4. 将文件重命名为 `gold_price_history.csv`，放在本目录下

### 方式 2: akshare（仅覆盖 2016 年至今）

自动获取上海黄金交易所数据，适用于俄乌战争分析。

---

In [ ]:
# 安装依赖（如需要）
# !pip install akshare plotly pandas numpy

In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✓ 库加载完成")

## 1. 加载战争事件数据

In [ ]:
with open('war_events.json', 'r', encoding='utf-8') as f:
    wars = json.load(f)

print("已加载战争事件数据：")
for war_id, war_info in wars.items():
    end_date = war_info['end'] if war_info['end'] else '至今'
    print(f"  • {war_info['name']}: {war_info['start']} ~ {end_date}")

## 2. 获取黄金价格数据

In [ ]:
# 检查数据源
csv_file = 'gold_price_history.csv'

if os.path.exists(csv_file):
    print(f"✓ 发现本地数据文件: {csv_file}")
    DATA_SOURCE = 'csv'
else:
    print("⚠ 未找到 gold_price_history.csv")
    print("  → 将使用 akshare 获取数据（仅覆盖 2016 年至今）")
    print("  → 如需完整历史数据，请从 Investing.com 下载 CSV")
    DATA_SOURCE = 'akshare'

print(f"\n当前数据源: {DATA_SOURCE}")

In [ ]:
def load_from_csv(filepath):
    """从 CSV 文件加载（Investing.com 格式）"""
    df = pd.read_csv(filepath)
    
    # Investing.com 格式: Date, Price, Open, High, Low, Vol., Change%
    col_map = {'Date': 'date', 'Price': 'close', 'Open': 'open', 'High': 'high', 'Low': 'low'}
    df = df.rename(columns=col_map)
    
    # 处理日期
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    # 清理价格（移除逗号）
    for col in ['close', 'open', 'high', 'low']:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].astype(str).str.replace(',', '').astype(float)
    
    return df[['date', 'close']].dropna()


def load_from_akshare():
    """从 akshare 获取上海黄金交易所数据"""
    import akshare as ak
    print("正在从 akshare 获取数据...")
    
    df = ak.spot_hist_sge(symbol='Au99.99')
    df = df[['date', 'close']].copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    return df


# 加载数据
if DATA_SOURCE == 'csv':
    df = load_from_csv(csv_file)
else:
    df = load_from_akshare()

print(f"\n✓ 数据加载完成")
print(f"  时间范围: {df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
print(f"  数据行数: {len(df):,}")
df.head()

### 数据覆盖检查

In [ ]:
print("战争数据覆盖情况：")
print("=" * 50)

data_start = df['date'].min()
data_end = df['date'].max()
available_wars = []

for war_id, war in wars.items():
    war_start = pd.to_datetime(war['start'])
    war_end = pd.to_datetime(war['end']) if war['end'] else data_end
    
    if war_end >= data_start and war_start <= data_end:
        overlap_start = max(war_start, data_start)
        overlap_end = min(war_end, data_end)
        
        if war['end']:
            coverage = (overlap_end - overlap_start).days / (war_end - war_start).days * 100
        else:
            coverage = 100
        
        status = "✓" if coverage > 10 else "⚠"
        print(f"{status} {war['name']}: {coverage:.0f}% 覆盖")
        available_wars.append(war_id)
    else:
        print(f"✗ {war['name']}: 无数据")

print(f"\n可分析战争数: {len(available_wars)}")

## 3. 分析函数

In [ ]:
def analyze_war(df, war_id, wars, days_before=90, days_after=180):
    """分析单个战争期间的黄金价格"""
    war = wars[war_id]
    start_date = pd.to_datetime(war['start'])
    end_date = pd.to_datetime(war['end']) if war['end'] else df['date'].max()
    
    window_start = start_date - timedelta(days=days_before)
    window_end = end_date + timedelta(days=days_after) if war['end'] else df['date'].max()
    
    mask = (df['date'] >= window_start) & (df['date'] <= window_end)
    war_df = df[mask].copy()
    
    if war_df.empty or len(war_df) < 5:
        return None
    
    # 基准价格（战争开始前）
    pre_mask = war_df['date'] < start_date
    base_price = war_df[pre_mask]['close'].iloc[-1] if pre_mask.any() else war_df['close'].iloc[0]
    
    war_df['pct_change'] = (war_df['close'] / base_price - 1) * 100
    
    # 战争期间统计
    during_mask = (war_df['date'] >= start_date) & (war_df['date'] <= end_date)
    during = war_df[during_mask]
    
    if during.empty:
        return None
    
    stats = {
        'war_name': war['name'],
        'name_en': war['name_en'],
        'color': war['color'],
        'base_price': base_price,
        'max_price': during['close'].max(),
        'min_price': during['close'].min(),
        'max_change': during['pct_change'].max(),
        'min_change': during['pct_change'].min(),
        'end_price': war_df[war_df['date'] <= end_date]['close'].iloc[-1],
        'end_change': war_df[war_df['date'] <= end_date]['pct_change'].iloc[-1],
    }
    
    return {'data': war_df, 'stats': stats, 'war_info': war, 'window': (window_start, window_end, start_date, end_date)}


def plot_war(result, title=None):
    """绘制战争期间价格图"""
    if result is None:
        return None
    
    df = result['data']
    war = result['war_info']
    _, _, war_start, war_end = result['window']
    
    fig = go.Figure()
    
    # 价格线
    fig.add_trace(go.Scatter(
        x=df['date'], y=df['close'],
        mode='lines', name='黄金价格',
        line=dict(color='#FFD700', width=2),
        hovertemplate='日期: %{x|%Y-%m-%d}<br>价格: $%{y:.2f}<extra></extra>'
    ))
    
    # 战争期间背景
    fig.add_vrect(
        x0=war_start, x1=war_end if war['end'] else df['date'].max(),
        fillcolor=war['color'], opacity=0.2, layer='below', line_width=0
    )
    
    # 事件标注
    for event in war['events']:
        event_date = pd.to_datetime(event['date'])
        if df['date'].min() <= event_date <= df['date'].max():
            idx = min(df['date'].searchsorted(event_date), len(df) - 1)
            price = df.iloc[idx]['close']
            
            colors = {'start': '#e74c3c', 'end': '#27ae60', 'military': '#f39c12', 'political': '#3498db'}
            
            fig.add_trace(go.Scatter(
                x=[event_date], y=[price],
                mode='markers', name=event['label'],
                marker=dict(size=10, color=colors.get(event['type'], '#95a5a6'), symbol='triangle-down'),
                hovertemplate=f'{event["label"]}<extra></extra>'
            ))
    
    fig.add_vline(x=war_start, line_dash='dash', line_color='red', opacity=0.7)
    
    fig.update_layout(
        title=title or f"{war['name']}期间黄金价格走势",
        xaxis_title='日期', yaxis_title='价格 (美元/盎司)',
        template='plotly_white', height=500, hovermode='x unified',
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    return fig

## 4. 俄乌战争分析

In [ ]:
result = analyze_war(df, 'russo_ukrainian_war', wars, days_before=180, days_after=365)
if result:
    print(f"=== {result['war_info']['name']} ===")
    for k, v in result['stats'].items():
        if isinstance(v, float):
            print(f"  {k}: {v:.2f}")
        elif isinstance(v, str) and k not in ['war_name', 'name_en', 'color']:
            print(f"  {k}: {v}")
    plot_war(result).show()
else:
    print("数据不足，无法分析")

## 5. 多战争对比

In [ ]:
def plot_comparison(df, wars, days_before=30, days_after=90):
    """多战争对比图（归一化）"""
    fig = go.Figure()
    
    for war_id, war in wars.items():
        start = pd.to_datetime(war['start'])
        end = pd.to_datetime(war['end']) if war['end'] else df['date'].max()
        
        window_start = start - timedelta(days=days_before)
        window_end = end + timedelta(days=days_after) if war['end'] else df['date'].max()
        
        mask = (df['date'] >= window_start) & (df['date'] <= window_end)
        war_df = df[mask].copy()
        
        if len(war_df) < 5:
            continue
        
        idx = min(war_df['date'].searchsorted(start), len(war_df) - 1)
        base = war_df.iloc[idx]['close']
        
        if pd.isna(base) or base == 0:
            continue
        
        war_df['normalized'] = (war_df['close'] / base) * 100
        war_df['days'] = (war_df['date'] - start).dt.days
        
        fig.add_trace(go.Scatter(
            x=war_df['days'], y=war_df['normalized'],
            mode='lines', name=war['name_en'],
            line=dict(color=war['color'], width=2)
        ))
    
    fig.add_vline(x=0, line_dash='dash', line_color='black', opacity=0.5)
    fig.add_hline(y=100, line_dash='dot', line_color='gray', opacity=0.5)
    
    fig.update_layout(
        title='历次战争期间黄金价格变化对比（战争开始日=100）',
        xaxis_title='距战争开始天数', yaxis_title='相对价格指数',
        template='plotly_white', height=600,
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    return fig

plot_comparison(df, wars, days_before=30, days_after=180).show()

## 6. 统计汇总

In [ ]:
summary = []
for war_id in wars.keys():
    result = analyze_war(df, war_id, wars, days_before=30, days_after=90)
    if result and result['stats']:
        summary.append(result['stats'])

if summary:
    summary_df = pd.DataFrame(summary).set_index('war_name')
    display(summary_df[['base_price', 'max_price', 'max_change', 'end_price', 'end_change']])
    
    # 可视化
    fig = make_subplots(rows=1, cols=2, subplot_titles=('最大涨幅 (%)', '结束时涨跌幅 (%)'))
    
    names = list(summary_df.index)
    colors = [s['color'] for s in summary]
    
    fig.add_trace(go.Bar(x=names, y=summary_df['max_change'], marker_color=colors), row=1, col=1)
    fig.add_trace(go.Bar(x=names, y=summary_df['end_change'], marker_color=colors), row=1, col=2)
    
    fig.update_layout(title_text='战争期间黄金表现汇总', showlegend=False, height=400, template='plotly_white')
    fig.update_xaxes(tickangle=45)
    fig.show()
else:
    print("没有足够的战争数据进行汇总")

## 7. 结论

In [ ]:
print("="*60)
print("黄金 vs 战争：关键发现")
print("="*60)

if summary:
    print(f"\n📊 统计摘要:")
    print(f"  • 平均最大涨幅: {summary_df['max_change'].mean():.1f}%")
    print(f"  • 平均结束时涨幅: {summary_df['end_change'].mean():.1f}%")
    print(f"\n📈 各战争表现:")
    for idx, row in summary_df.iterrows():
        print(f"  • {idx}: 最大 {row['max_change']:+.1f}%, 结束 {row['end_change']:+.1f}%")
else:
    print("\n⚠ 数据不足，请下载完整的历史数据 CSV 文件")
    print("\n下载步骤:")
    print("1. 访问 https://www.investing.com/commodities/gold-historical-data")
    print("2. 设置日期范围: 1970-01-01 至今")
    print("3. 下载并保存为 gold_price_history.csv")